# 02. Multi-Head Attention + 풀 Transformer Block

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_03_transformer/02_multihead_and_transformer_block.ipynb)

이전 노트북(`01_attention_from_scratch.ipynb`) 에서 본 단일 scaled dot-product attention 을
**multi-head** 로 확장하고, 그 위에 **positional encoding / LayerNorm / Residual / FFN** 을 얹어
"Attention is All You Need" 의 풀 인코더/디코더 block 까지 직접 짠다.

흐름:
1. Multi-head attention — 왜 "여러 개" 인가
2. 수식과 numpy 직접 구현
3. PyTorch `nn.MultiheadAttention` 과 결과 일치 검증
4. Positional Encoding (sinusoidal)
5. Layer Normalization vs Batch Normalization
6. FFN (position-wise feed-forward)
7. Residual connection + Pre-LN vs Post-LN
8. 풀 Transformer Encoder Block
9. Decoder Block (causal self-attn + cross-attn)
10. PyTorch `nn.TransformerEncoderLayer` 와 비교

> 이 노트북을 끝내면 PyTorch `nn.Transformer` 가 무엇을 캡슐화한 한 줄인지가 보입니다.
> 다음 `03_huggingface_practical.ipynb` 에서 실제 사전학습 모델 (BERT/GPT/한국어 모델) 을 사용합니다.


In [ ]:
# 공통 실행 환경 준비 (Colab/Local 통합)
from pathlib import Path
import subprocess, sys, os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_03_transformer'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

try:
    import torch  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa


## 1. 왜 Multi-Head 인가

단일 attention 의 한계:

> 한 번의 softmax 가중 평균은 "한 가지 종류의 관계" 만 잡는다.

언어에서 한 문장의 토큰들은 **여러 차원의 관계** 를 동시에 가진다:

- 문법적 의존 (주어-동사)
- 공지시 (대명사-선행사)
- 어휘적 유사성
- 위치적 인접성

**Multi-head 의 발상**: $d_\text{model}$ 차원을 $h$ 개의 부분공간으로 쪼개고, 각 head 가 *독립적으로* attention 을 수행해
서로 다른 관계를 동시에 잡게 한다.

수식:

$$
\mathrm{MultiHead}(Q,K,V) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\, W^O
$$
$$
\mathrm{head}_i = \mathrm{Attention}(Q W^Q_i,\; K W^K_i,\; V W^V_i)
$$

여기서 각 $W^{Q,K,V}_i \in \mathbb{R}^{d_\text{model} \times d_k}$, $W^O \in \mathbb{R}^{(h \cdot d_v) \times d_\text{model}}$.
보통 $d_k = d_v = d_\text{model} / h$ 로 잡아서 head 가 늘어도 총 파라미터 수는 일정.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

np.random.seed(0)
torch.manual_seed(0)


def softmax_np(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def attention_np(Q, K, V, mask=None):
    """01 노트북에서 만든 scaled dot-product attention."""
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    w = softmax_np(scores, axis=-1)
    return w @ V, w


## 2. Multi-Head Attention 직접 구현

핵심 트릭: 머리(head) 별로 따로 행렬곱 하지 않고, **한 번의 큰 선형변환 후 reshape** 으로 처리한다.
이게 GPU 친화적이고 실제 라이브러리 구현과도 같다.

배치를 무시한 한 시퀀스 ($N$ 토큰) 에 대해:

1. $Q, K, V \in \mathbb{R}^{N \times d_\text{model}}$ 를 $W^{Q,K,V}$ 한 번에 적용 (이미 들어왔다고 가정).
2. $(N, d_\text{model})$ → reshape → $(N, h, d_k)$ → transpose → $(h, N, d_k)$.
3. head 차원에 대해 한꺼번에 attention.
4. 결과 $(h, N, d_v)$ → 다시 transpose/reshape → $(N, h \cdot d_v) = (N, d_\text{model})$.
5. $W^O$ 한 번 더 곱해 출력.


In [ ]:
def multihead_attention_np(X, Wq, Wk, Wv, Wo, num_heads, mask=None):
    """단일 시퀀스에 대한 multi-head self-attention.

    Args:
        X:  (N, d_model)
        Wq, Wk, Wv: (d_model, d_model)
        Wo: (d_model, d_model)
        num_heads: h
    """
    N, d_model = X.shape
    h = num_heads
    assert d_model % h == 0, 'd_model 은 head 수로 나눠떨어져야 함'
    d_k = d_model // h

    # 1) 큰 선형변환: 한 번에 Q, K, V 만들기
    Q = X @ Wq                                   # (N, d_model)
    K = X @ Wk
    V = X @ Wv

    # 2) head 분리: (N, d_model) -> (N, h, d_k) -> (h, N, d_k)
    def split_heads(t):
        return t.reshape(N, h, d_k).transpose(1, 0, 2)

    Qh, Kh, Vh = split_heads(Q), split_heads(K), split_heads(V)

    # 3) head 차원 통째로 attention (broadcasting)
    out_h, weights_h = attention_np(Qh, Kh, Vh, mask=mask)   # (h, N, d_k)

    # 4) head 합치기: (h, N, d_k) -> (N, h, d_k) -> (N, d_model)
    out_concat = out_h.transpose(1, 0, 2).reshape(N, d_model)

    # 5) 출력 사영
    out = out_concat @ Wo
    return out, weights_h


# 작은 예시
N, d_model, h = 6, 32, 4
X = np.random.randn(N, d_model).astype(np.float32)
Wq = np.random.randn(d_model, d_model).astype(np.float32) * 0.1
Wk = np.random.randn(d_model, d_model).astype(np.float32) * 0.1
Wv = np.random.randn(d_model, d_model).astype(np.float32) * 0.1
Wo = np.random.randn(d_model, d_model).astype(np.float32) * 0.1

out_np, w_np = multihead_attention_np(X, Wq, Wk, Wv, Wo, num_heads=h)
print('output shape :', out_np.shape, '  (N, d_model)')
print('weights shape:', w_np.shape, '  (h, N, N) — 각 head 마다 attention 행렬')


## 3. PyTorch `nn.MultiheadAttention` 과 결과 일치 검증

PyTorch 의 nn.MultiheadAttention 은 기본 `(L, N, E)` (시퀀스 우선) 입력을 받는다 (`batch_first=True` 옵션 가능).
같은 가중치를 주입해서 우리 numpy 구현과 결과가 일치하는지 확인한다.


In [ ]:
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=h, bias=False, batch_first=True)

# nn.MultiheadAttention 의 in_proj_weight 는 (3*d_model, d_model) 로 [Wq; Wk; Wv] 를 행 방향 concat.
# 우리 numpy 가중치를 그대로 주입.
with torch.no_grad():
    mha.in_proj_weight.copy_(torch.from_numpy(np.concatenate([Wq.T, Wk.T, Wv.T], axis=0)))
    mha.out_proj.weight.copy_(torch.from_numpy(Wo.T))

Xt = torch.from_numpy(X).unsqueeze(0)            # (1, N, d_model)
out_t, _ = mha(Xt, Xt, Xt, need_weights=False)
out_torch = out_t.squeeze(0).detach().numpy()

print('일치 :', np.allclose(out_np, out_torch, atol=1e-4))
print('최대 오차:', np.max(np.abs(out_np - out_torch)))


## 4. Positional Encoding — "순서" 를 어떻게 알려주나

Attention 자체는 **순서 무관 (permutation-equivariant)** 이다. 토큰 순서를 바꿔도 같은 출력이 나온다.
언어/시계열에서 순서는 핵심 정보이므로, **위치 자체를 임베딩에 더해 줘야** 한다.

원 논문의 sinusoidal positional encoding:

$$
\mathrm{PE}(\text{pos}, 2i)   = \sin\!\left(\frac{\text{pos}}{10000^{2i/d_\text{model}}}\right)
$$
$$
\mathrm{PE}(\text{pos}, 2i+1) = \cos\!\left(\frac{\text{pos}}{10000^{2i/d_\text{model}}}\right)
$$

장점:
- 학습 파라미터 없이 임의 길이까지 외삽 가능.
- 두 위치의 차이가 선형변환으로 표현 가능 (상대 위치 정보를 모델이 쉽게 학습).

> 최근 모델은 학습된 positional embedding 또는 RoPE / ALiBi 같은 변형을 더 많이 쓴다. 여기서는 원전을 따른다.


In [ ]:
def sinusoidal_positional_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """위 수식의 numpy 구현. shape: (seq_len, d_model)."""
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    pos = np.arange(seq_len)[:, None]                              # (L, 1)
    i = np.arange(d_model)[None, :]                                # (1, D)
    angle_rates = 1.0 / np.power(10000.0, (2 * (i // 2)) / d_model)
    angles = pos * angle_rates                                     # (L, D)
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


pe = sinusoidal_positional_encoding(seq_len=64, d_model=32)
print('PE shape:', pe.shape)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3))
plt.imshow(pe, aspect='auto', cmap='RdBu')
plt.xlabel('feature dim'); plt.ylabel('position'); plt.title('Sinusoidal Positional Encoding')
plt.colorbar()
plt.tight_layout()
plt.show()


## 5. LayerNorm vs BatchNorm — Transformer 가 LN 을 쓰는 이유

- **BatchNorm**: 배치 차원에 대해 평균/분산 정규화. CNN 에서 표준. 그러나 시퀀스 길이가 가변이고 배치가 작은 NLP 환경에서는 통계가 불안정.
- **LayerNorm**: 한 샘플 안에서 *feature 차원* 에 대해 정규화. 배치/시퀀스 길이에 독립적 → transformer 에 적합.

수식 (한 토큰 벡터 $x \in \mathbb{R}^{d}$ 에 대해):

$$
\mu = \frac{1}{d}\sum_i x_i,\quad
\sigma^2 = \frac{1}{d}\sum_i (x_i - \mu)^2
$$
$$
\mathrm{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

$\gamma, \beta$ 는 학습 가능한 스케일/바이어스.


In [ ]:
ln = nn.LayerNorm(d_model)
y = ln(Xt)
print('LN 후 평균(거의 0):', y.mean(dim=-1)[0, :3].detach().numpy())
print('LN 후 분산(거의 1):', y.var(dim=-1, unbiased=False)[0, :3].detach().numpy())


## 6. Position-wise Feed-Forward Network (FFN)

각 토큰 위치에 **독립적으로** 적용되는 작은 MLP. 토큰끼리 상호작용은 attention 이 담당하고,
"한 토큰 안의 비선형 변환" 은 FFN 이 담당한다.

$$
\mathrm{FFN}(x) = \mathrm{Linear}_2(\mathrm{ReLU}(\mathrm{Linear}_1(x)))
$$

원 논문에서는 hidden 크기를 $d_\text{ff} = 4 \cdot d_\text{model}$ 로 잡는다. 모델 파라미터의 큰 비중이 여기에 있다.


In [ ]:
class FFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


ffn = FFN(d_model=d_model)
print('FFN 출력 shape:', ffn(Xt).shape)
print('파라미터 수    :', sum(p.numel() for p in ffn.parameters()))


## 7. Residual Connection + Pre-LN vs Post-LN

각 sub-layer (attention, FFN) 의 입력 $x$ 와 출력을 **더해서** 다음 층으로 보낸다 (skip connection):

$$ x \leftarrow x + \mathrm{Sublayer}(x) $$

장점:
- gradient 가 layer 를 "건너뛰어" 흐를 수 있어 깊은 네트워크가 학습 가능.
- 항등 함수 학습이 자명 → "필요할 때만" 변환을 학습.

LayerNorm 위치에 따라 두 가지 스타일:

- **Post-LN** (원 논문): $x + \mathrm{Sublayer}(x)$ 후에 LN. 학습 초반 불안정하지만 수렴 후 약간 더 좋다고 알려짐.
- **Pre-LN** (실용적): LN 먼저 → Sublayer → residual. **학습이 훨씬 안정적이라 최근 표준.**

$$
\text{Post-LN}: \quad y = \mathrm{LN}(x + \mathrm{Sublayer}(x)) \\
\text{Pre-LN }: \quad y = x + \mathrm{Sublayer}(\mathrm{LN}(x))
$$


## 8. 풀 Transformer Encoder Block (Pre-LN)

위 부품들 (Multi-Head Self-Attention + LN + Residual + FFN + LN + Residual) 을 합치면
**transformer 인코더 한 층** 이다. BERT 는 이 층을 12 (base) ~ 24 (large) 개 쌓은 것에 불과하다.


In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Pre-LN 스타일 transformer encoder block.

    구조:
        x -> LN -> MHA(self-attn) -> + x  (residual)
          -> LN -> FFN              -> + x  (residual)
    """
    def __init__(self, d_model: int, num_heads: int, d_ff: int = None, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attn_mask=None, key_padding_mask=None) -> torch.Tensor:
        # 1) Self-attention sub-layer (pre-LN)
        h = self.ln1(x)
        attn_out, _ = self.mha(h, h, h, attn_mask=attn_mask,
                               key_padding_mask=key_padding_mask, need_weights=False)
        x = x + self.drop(attn_out)
        # 2) FFN sub-layer (pre-LN)
        h = self.ln2(x)
        x = x + self.drop(self.ffn(h))
        return x


block = TransformerEncoderBlock(d_model=d_model, num_heads=h)
y = block(Xt)
print('block 출력 shape:', y.shape)
print('block 파라미터 수:', sum(p.numel() for p in block.parameters()))


## 9. Decoder Block — Causal Self-Attn + Cross-Attn

언어모델/번역 디코더는 인코더 블록에 두 가지가 추가된다:

1. **Causal mask** 가 적용된 self-attention (미래 토큰을 보면 안 됨)
2. **Cross-attention** : Q 는 디코더 자신, K/V 는 **인코더 출력**. "입력 문장의 어느 부분을 볼지" 를 학습.

GPT 같은 **decoder-only** 모델은 cross-attention 을 빼고 causal self-attn 만 쌓는다 (입력 = 자기 자신의 과거).


In [ ]:
class TransformerDecoderBlock(nn.Module):
    """Causal self-attn + cross-attn + FFN. Pre-LN."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int = None, dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mem: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        # causal mask: True = 차단. PyTorch 의 attn_mask 는 True 가 차단.
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)

        # 1) Causal self-attention
        h = self.ln1(x)
        a, _ = self.self_attn(h, h, h, attn_mask=causal, need_weights=False)
        x = x + self.drop(a)
        # 2) Cross-attention: Q = 디코더, K/V = 인코더 출력 (mem)
        h = self.ln2(x)
        a, _ = self.cross_attn(h, mem, mem, need_weights=False)
        x = x + self.drop(a)
        # 3) FFN
        h = self.ln3(x)
        x = x + self.drop(self.ffn(h))
        return x


# 인코더 출력 (예: 길이 8) 을 디코더 (길이 5) 가 참조
mem = torch.randn(1, 8, d_model)
dec_in = torch.randn(1, 5, d_model)
decoder_block = TransformerDecoderBlock(d_model, h)
print('decoder block 출력 shape:', decoder_block(dec_in, mem).shape)


## 10. PyTorch `nn.TransformerEncoderLayer` 와 비교

PyTorch 의 한 줄 인코더 층은 위에서 만든 `TransformerEncoderBlock` 과 동일한 구조 (기본은 Post-LN, `norm_first=True` 로 Pre-LN).
파라미터 수가 같은지 확인한다.


In [ ]:
official = nn.TransformerEncoderLayer(
    d_model=d_model, nhead=h, dim_feedforward=4 * d_model,
    dropout=0.0, batch_first=True, norm_first=True,
)
print('official 파라미터:', sum(p.numel() for p in official.parameters()))
print('우리 block 파라미터:', sum(p.numel() for p in block.parameters()))
print('출력 shape       :', official(Xt).shape)


## 11. 흔한 함정 (Common Pitfalls)

1. **head 차원과 d_model**: `d_model % num_heads != 0` 이면 분할 불가 → assert 로 차단.
2. **mask shape/타입**: PyTorch `attn_mask` 는 `(L, L)` 또는 `(B*h, L, L)`. `key_padding_mask` 는 `(B, L)` 별도.
3. **batch_first**: 기본은 `(L, B, E)` (시퀀스 우선). 실수의 단골. 항상 `batch_first=True` 로 통일하는 걸 권장.
4. **PE 더하기 vs 이어붙이기**: 원 논문은 **더하기**. 같은 차원으로 만들어 임베딩에 element-wise add.
5. **Pre-LN vs Post-LN 혼용 금지**: 한 모델 안에서 일관 유지. Pre-LN 이 학습 안정성 면에서 사실상 표준.
6. **Dropout 위치**: 보통 attention 출력과 FFN 출력에 적용. 너무 많이 넣으면 학습 느려짐.


## 정리

오늘 만든 부품을 한 그림으로:

```
                  [Input Embedding] + [Positional Encoding]
                                |
                                v
        ┌───────────────────────────────────────────────┐
        │  Pre-LN Transformer Encoder Block × N         │
        │   ┌──── x ────┐                              │
        │   │ LN → MHA(self) → +residual              │
        │   │ LN → FFN        → +residual              │
        │   └────────────┘                              │
        └───────────────────────────────────────────────┘
                                |
                                v
                       (출력 표현 → 분류, 생성 등)
```

- **GPT** = decoder-only (causal self-attn + FFN), N 층
- **BERT** = encoder-only (양방향 self-attn + FFN), N 층
- **원 Transformer (번역)** = encoder + decoder, decoder 에 cross-attn 추가

다음 노트북 `03_huggingface_practical.ipynb` 에서:
- HuggingFace 의 사전학습 모델 로딩 / 추론 / fine-tuning 실전
- BERT / GPT / 한국어 모델 (klue/bert-base, beomi/kcbert 등) 사용 예
- LLM 시대로 어떻게 이어지는지
